# LBG without degrade

In [1]:
import numpy as np
import pandas as pd
from astropy.coordinates import SkyCoord
import astropy.units as u


# ============================================================
# FILES
# ============================================================

i_file = "/Users/aishwarya/Desktop/new_cdfs/cat/LBG_I_degraded.cat"
z_file = "/Users/aishwarya/Desktop/new_cdfs/cat/LBG_Z_degraded.cat"
y_file = "/Users/aishwarya/Desktop/new_cdfs/cat/LBG_Y_degraded.cat"

redlist_file = "/Users/aishwarya/Downloads/candidates_red_list_cdfs.txt"


# ============================================================
# COLUMN FORMAT (SExtractor style)
# ============================================================

colnames = [
    "ID", "X", "Y", "RA", "DEC",
    "MAG_APER", "MAGERR_APER",
    "MAG_AUTO", "MAGERR_AUTO", "FLAGS"
]


# ============================================================
# LOAD CATALOGS (robust whitespace parsing)
# ============================================================

i_df = pd.read_csv(i_file, delim_whitespace=True, names=colnames)
z_df = pd.read_csv(z_file, delim_whitespace=True, names=colnames)
y_df = pd.read_csv(y_file, delim_whitespace=True, names=colnames)

print("Original catalog sizes:")
print("I:", len(i_df))
print("Z:", len(z_df))
print("Y:", len(y_df))


# ============================================================
# CLEAN BROKEN ROWS
# ============================================================

def clean_catalog(df):

    df = df.replace([np.inf, -np.inf], np.nan)

    df = df.dropna(subset=["RA", "DEC", "MAG_APER", "MAGERR_APER"])

    df = df[
        (df.RA >= 0) & (df.RA <= 360) &
        (df.DEC >= -90) & (df.DEC <= 90)
    ]

    return df.reset_index(drop=True)


i_df = clean_catalog(i_df)
z_df = clean_catalog(z_df)
y_df = clean_catalog(y_df)

print("\nCatalog sizes after cleaning:")
print("I:", len(i_df))
print("Z:", len(z_df))
print("Y:", len(y_df))


# ============================================================
# ZEROPOINT CALIBRATION (using degraded mags)
# ============================================================

ZP = {
    "i": (31.354,  0.004),
    "z": (31.524,  0.004),
    "y": (30.3382, 0.009)
}


def calibrate(df, band):

    zp, zp_err = ZP[band]

    df["MAG_CAL"] = df["MAG_APER"] + zp
    df["MAGERR_CAL"] = np.sqrt(df["MAGERR_APER"]**2 + zp_err**2)


calibrate(i_df, "i")
calibrate(z_df, "z")
calibrate(y_df, "y")


# ============================================================
# SNR FROM MAG ERROR
# ============================================================

def magerr_to_snr(err):

    snr = 2.5 / (np.log(10) * err)

    snr[~np.isfinite(snr)] = 0

    return snr


# ============================================================
# COORDINATE MATCHING
# ============================================================

y_coords = SkyCoord(y_df.RA.values * u.deg,
                    y_df.DEC.values * u.deg)

i_coords = SkyCoord(i_df.RA.values * u.deg,
                    i_df.DEC.values * u.deg)

z_coords = SkyCoord(z_df.RA.values * u.deg,
                    z_df.DEC.values * u.deg)

match_radius = 1.0 * u.arcsec

idx_i, sep_i, _ = y_coords.match_to_catalog_sky(i_coords)
idx_z, sep_z, _ = y_coords.match_to_catalog_sky(z_coords)


def matched_array(src_df, idx, sep):

    n = len(y_df)

    mag = np.full(n, np.nan)
    err = np.full(n, np.nan)

    detected = sep < match_radius

    mag[detected] = src_df.MAG_CAL.values[idx[detected]]
    err[detected] = src_df.MAGERR_CAL.values[idx[detected]]

    return mag, err, detected


# ============================================================
# MATCHED PHOTOMETRY
# ============================================================

i_mag, i_err, i_detected = matched_array(i_df, idx_i, sep_i)
z_mag, z_err, z_detected = matched_array(z_df, idx_z, sep_z)

y_mag = y_df.MAG_CAL.values
y_err = y_df.MAGERR_CAL.values


# ============================================================
# SNR
# ============================================================

i_snr = magerr_to_snr(i_err)
z_snr = magerr_to_snr(z_err)
y_snr = magerr_to_snr(y_err)


# ============================================================
# I-BAND NON-DETECTION HANDLING
# ============================================================

I_LIMIT = 27.84

i_nondet = (~i_detected) | (i_snr < 2.0)

i_mag[i_nondet] = I_LIMIT
i_err[i_nondet] = 0.0


# ============================================================
# COLORS
# ============================================================

color_zy = z_mag - y_mag
color_iz = i_mag - z_mag

color_zy_err = np.sqrt(z_err**2 + y_err**2)


# ============================================================
# SELECTION CUTS
# ============================================================

cut_z_detected = z_detected
cut_z_snr = z_snr > 4.0
cut_y_snr = y_snr > 4.0

cut_zy = color_zy > 0.8
cut_break = color_iz > 1.0

cut_i_dropout = i_nondet
cut_sig = np.abs(color_zy) > 2.5 * color_zy_err


# ============================================================
# FINAL SELECTION
# ============================================================

final_sel = (
    cut_z_detected &
    cut_z_snr &
    cut_y_snr &
    cut_zy &
    cut_break &
    cut_i_dropout &
    cut_sig
)

lbg_idx = np.where(final_sel)[0]

print(f"\nAfter selection cuts: {len(lbg_idx)} candidates")


# ============================================================
# BUILD CANDIDATE CATALOG
# ============================================================

lbg_candidates = pd.DataFrame({

    "RA": y_df.RA.values[lbg_idx],
    "DEC": y_df.DEC.values[lbg_idx],

    "Y_mag": y_mag[lbg_idx],
    "Y_err": y_err[lbg_idx],

    "Z_mag": z_mag[lbg_idx],
    "Z_err": z_err[lbg_idx],
    "Z_SNR": z_snr[lbg_idx],

    "I_mag": i_mag[lbg_idx],
    "I_err": i_err[lbg_idx],
    "I_SNR": i_snr[lbg_idx],

    "Z-Y": color_zy[lbg_idx],
    "I-Z": color_iz[lbg_idx],

    "I_detected": i_detected[lbg_idx]
})


# ============================================================
# REMOVE RED-LIST SOURCES
# ============================================================

red_df = pd.read_csv(redlist_file, delim_whitespace=True,
                     names=["RA", "DEC"])

red_coords = SkyCoord(red_df.RA.values * u.deg,
                      red_df.DEC.values * u.deg)

lbg_coords = SkyCoord(lbg_candidates.RA.values * u.deg,
                      lbg_candidates.DEC.values * u.deg)

idx_red, sep_red, _ = lbg_coords.match_to_catalog_sky(red_coords)

lbg_candidates = lbg_candidates[sep_red.arcsec > 1.0]


# ============================================================
# OUTPUT
# ============================================================

outpath = "/Users/aishwarya/Documents/Lyman_alpha/CDFS_candidates_ZY.cat"

lbg_candidates.to_csv(outpath, index=False)

print(f"\nFinal candidates after red-list removal: {len(lbg_candidates)}")
print(f"Saved to {outpath}")


print("\nFinal candidates:")

for _, row in lbg_candidates.iterrows():

    print(
        f"RA={row.RA:.6f}, DEC={row.DEC:.6f}, "
        f"I_SNR={row.I_SNR:.2f}, "
        f"I_mag={row.I_mag:.2f}, "
        f"z_mag={row.Z_mag:.2f}, "
        f"y_mag={row.Y_mag:.2f}"
    )

/var/folders/9v/db3j63px68q3f33kdb7_7wbc0000gn/T/ipykernel_2743/1811513055.py:33: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  i_df = pd.read_csv(i_file, delim_whitespace=True, names=colnames)
/var/folders/9v/db3j63px68q3f33kdb7_7wbc0000gn/T/ipykernel_2743/1811513055.py:34: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  z_df = pd.read_csv(z_file, delim_whitespace=True, names=colnames)
/var/folders/9v/db3j63px68q3f33kdb7_7wbc0000gn/T/ipykernel_2743/1811513055.py:35: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  y_df = pd.read_csv(y_file, delim_whitespace=True, names=colnames)


Original catalog sizes:
I: 27804
Z: 172864
Y: 178963

Catalog sizes after cleaning:
I: 15
Z: 366
Y: 65

After selection cuts: 0 candidates

Final candidates after red-list removal: 0
Saved to /Users/aishwarya/Documents/Lyman_alpha/CDFS_candidates_ZY.cat

Final candidates:


/var/folders/9v/db3j63px68q3f33kdb7_7wbc0000gn/T/ipykernel_2743/1811513055.py:249: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  red_df = pd.read_csv(redlist_file, delim_whitespace=True,


# LBG with degrade

In [3]:
import numpy as np
import pandas as pd
from astropy.coordinates import SkyCoord
import astropy.units as u

# ============================================================
# File paths
# ============================================================
i_file = "/Users/aishwarya/Desktop/Lyman_alpha_2/CAT/i_band_degraded.cat"
z_file = "/Users/aishwarya/Desktop/Lyman_alpha_2/CAT/z_band_degraded.cat"
y_file = "/Users/aishwarya/Desktop/new_cdfs/cat/LBG_Y_degraded.cat"
redlist_file = "/Users/aishwarya/Downloads/candidates_red_list_cdfs.txt"

# ============================================================
# READ ALL FILES (NO HEADER)
# ============================================================
i_df = pd.read_csv(i_file, sep=r"\s+", comment="#", header=None)
z_df = pd.read_csv(z_file, sep=r"\s+", comment="#", header=None)
y_df = pd.read_csv(y_file, sep=r"\s+", comment="#", header=None)

print("i shape:", i_df.shape)
print("z shape:", z_df.shape)
print("y shape:", y_df.shape)

# ============================================================
# ASSIGN COLUMN NAMES
# ============================================================

# i and z degraded catalogs (12 columns)
i_df.columns = [
    "NUMBER", "X_IMAGE", "Y_IMAGE", "RA", "DEC",
    "MAG_ORIG", "MAGERR_ORIG",
    "MAG_APER", "MAGERR_APER",
    "FLAGS",
    "MAG_DEGRADED", "MAGERR_DEGRADED"
]

z_df.columns = [
    "NUMBER", "X_IMAGE", "Y_IMAGE", "RA", "DEC",
    "MAG_ORIG", "MAGERR_ORIG",
    "MAG_APER", "MAGERR_APER",
    "FLAGS",
    "MAG_DEGRADED", "MAGERR_DEGRADED"
]

# Y catalog (10 columns from SExtractor)
y_df.columns = [
    "NUMBER", "X_IMAGE", "Y_IMAGE", "RA", "DEC",
    "MAG_ORIG", "MAGERR_ORIG",
    "MAG_APER", "MAGERR_APER",
    "FLAGS",
    "MAG_DEGRADED", "MAGERR_DEGRADED"
]

print(f"Y catalog size: {len(y_df)}")

# ============================================================
# USE DEGRADED MAGNITUDES (i & z)
# ============================================================
i_df["MAG_CAL"] = i_df["MAG_DEGRADED"]
i_df["MAGERR_CAL"] = i_df["MAGERR_DEGRADED"]


z_df["MAG_CAL"] = z_df["MAG_DEGRADED"]
z_df["MAGERR_CAL"] = z_df["MAGERR_DEGRADED"]

y_df["MAG_CAL"] = y_df["MAG_DEGRADED"]
y_df["MAGERR_CAL"] = y_df["MAGERR_DEGRADED"]


# ============================================================
# SNR from magnitude error
# ============================================================
def magerr_to_snr(err):
    snr = 2.5 / (np.log(10) * err)
    snr[~np.isfinite(snr)] = 0.0
    return snr

# ============================================================
# Coordinate matching (Y is reference)
# ============================================================

y_coords = SkyCoord(y_df["RA"].values, y_df["DEC"].values, unit="deg")

i_coords = SkyCoord(i_df["RA"].values, i_df["DEC"].values, unit="deg")
z_coords = SkyCoord(z_df["RA"].values, z_df["DEC"].values, unit="deg")

match_radius = 1.0 * u.arcsec

idx_i, sep_i, _ = y_coords.match_to_catalog_sky(i_coords)
idx_z, sep_z, _ = y_coords.match_to_catalog_sky(z_coords)
idx_y, sep_y, _ = y_coords.match_to_catalog_sky(y_coords)

def matched_array(src_df, idx, sep):
    n = len(y_df)
    mag = np.full(n, np.nan)
    err = np.full(n, np.nan)
    detected = sep < match_radius

    mag[detected] = src_df["MAG_CAL"].values[idx[detected]]
    err[detected] = src_df["MAGERR_CAL"].values[idx[detected]]

    return mag, err, detected

# ============================================================
# Matched photometry
# ============================================================
i_mag, i_err, i_detected = matched_array(i_df, idx_i, sep_i)
z_mag, z_err, z_detected = matched_array(z_df, idx_z, sep_z)
y_mag, y_err, y_detected = matched_array(y_df, idx_y, sep_y)



# ============================================================
# Compute SNR
# ============================================================
i_snr = magerr_to_snr(i_err)
z_snr = magerr_to_snr(z_err)
y_snr = magerr_to_snr(y_err)

# ============================================================
# FORCE I-band non-detections
# ============================================================
I_LIMIT = 27.84

i_nondet = (~i_detected) | (i_snr < 2.0)

i_mag[i_nondet] = I_LIMIT
i_err[i_nondet] = 0.0

# ============================================================
# Colors
# ============================================================
color_zy = z_mag - y_mag
color_iz = i_mag - z_mag
color_zy_err = np.sqrt(z_err**2 + y_err**2)

# ============================================================
# Selection cuts
# ============================================================
cut_z_detected = z_detected
cut_z_snr      = z_snr > 4.0
cut_y_snr      = y_snr > 4.0

cut_zy         = color_zy > 0.8
cut_break      = color_iz > 1.0

cut_i_dropout  = i_nondet
cut_sig        = np.abs(color_zy) > 2.5 * color_zy_err

# ============================================================
# Final selection
# ============================================================
final_sel = (
    cut_z_detected &
    cut_z_snr &
    cut_y_snr &
    cut_zy &
    cut_break &
    cut_i_dropout &
    cut_sig
)

lbg_idx = np.where(final_sel)[0]
print(f"After SNR-only cuts: {len(lbg_idx)} candidates")

# ============================================================
# Build LBG catalog
# ============================================================
lbg_candidates = lbg_candidates = pd.DataFrame({
    "RA": y_df["RA"].values[lbg_idx],
    "DEC": y_df["DEC"].values[lbg_idx],
    "Y_mag": y_mag[lbg_idx],
    "Y_err": y_err[lbg_idx],
    "Z_mag": z_mag[lbg_idx],
    "Z_err": z_err[lbg_idx],
    "Z_SNR": z_snr[lbg_idx],
    "I_mag": i_mag[lbg_idx],
    "I_err": i_err[lbg_idx],
    "I_SNR": i_snr[lbg_idx],
    "Z-Y": color_zy[lbg_idx],
    "I-Z": color_iz[lbg_idx],
    "I_detected": i_detected[lbg_idx]
})

# ============================================================
# Remove red-list sources
# ============================================================
red_df = pd.read_csv(redlist_file, sep=r"\s+", names=["RA", "DEC"])
red_coords = SkyCoord(red_df["RA"].values, red_df["DEC"].values, unit="deg")

lbg_coords = SkyCoord(
    lbg_candidates["RA"].values,
    lbg_candidates["DEC"].values,
    unit="deg"
)

idx_red, sep_red, _ = lbg_coords.match_to_catalog_sky(red_coords)
lbg_candidates = lbg_candidates[sep_red.arcsec > 1.0]

# ============================================================
# Output
# ============================================================
outpath = "/Users/aishwarya/Documents/Lyman_alpha/CDFS_candidates_ZY_degraded.cat"
lbg_candidates.to_csv(outpath, index=False)

print(f"Final LBG candidates after red-list removal: {len(lbg_candidates)}")
print(f"Saved to {outpath}")
# ============================================================
# Write DS9 region files
# ============================================================

# -------- 1) Circle regions (1 arcsec radius) --------
reg_arcsec_path = outpath.replace(".cat", "_circles_1arcsec.reg")

with open(reg_arcsec_path, "w") as f:
    f.write("# Region file format: DS9 version 4.1\n")
    f.write("global color=red width=2\n")
    f.write("fk5\n")

    for _, row in lbg_candidates.iterrows():
        ra = row["RA"]
        dec = row["DEC"]
        f.write(f'circle({ra},{dec},1")\n')

print(f"Arcsec circle region file saved to {reg_arcsec_path}")


# -------- 2) Point regions --------
reg_point_path = outpath.replace(".cat", "_points.reg")

with open(reg_point_path, "w") as f:
    f.write("# Region file format: DS9 version 4.1\n")
    f.write("global color=cyan point=cross 8 width=2\n")
    f.write("fk5\n")

    for _, row in lbg_candidates.iterrows():
        ra = row["RA"]
        dec = row["DEC"]
        f.write(f"point({ra},{dec})\n")

print(f"Point region file saved to {reg_point_path}")

i shape: (239300, 12)
z shape: (297003, 12)
y shape: (154924, 12)
Y catalog size: 154924
After SNR-only cuts: 2407 candidates
Final LBG candidates after red-list removal: 2402
Saved to /Users/aishwarya/Documents/Lyman_alpha/CDFS_candidates_ZY_degraded.cat
Arcsec circle region file saved to /Users/aishwarya/Documents/Lyman_alpha/CDFS_candidates_ZY_degraded_circles_1arcsec.reg
Point region file saved to /Users/aishwarya/Documents/Lyman_alpha/CDFS_candidates_ZY_degraded_points.reg


In [11]:
from astropy.io import fits
import numpy as np

data = fits.getdata("/Users/aishwarya/Desktop/new_cdfs/trimmed_2deg/trim2deg_y_sci_polygonmasked.fits")

print(np.nanmedian(data))
print(np.nanmean(data))
print(np.nanstd(data))

0.0
-0.4622995
9.560136


In [25]:
import numpy as np
import pandas as pd
from astropy.coordinates import SkyCoord
import astropy.units as u

# ============================================================
# File paths
# ============================================================
i_file = "/Users/aishwarya/Desktop/new_cdfs/cat/LBG_I_degraded.cat"
z_file = "/Users/aishwarya/Desktop/new_cdfs/cat/LBG_Z_degraded.cat"
y_file = "/Users/aishwarya/Desktop/new_cdfs/cat/LBG_Y_degraded.cat"
redlist_file = "/Users/aishwarya/Downloads/candidates_red_list_cdfs.txt"

# ============================================================
# READ FILES
# ============================================================
i_df = pd.read_csv(i_file, sep=r"\s+", comment="#", header=None)
z_df = pd.read_csv(z_file, sep=r"\s+", comment="#", header=None)
y_df = pd.read_csv(y_file, sep=r"\s+", comment="#", header=None)

print("i shape:", i_df.shape)
print("z shape:", z_df.shape)
print("y shape:", y_df.shape)

# ============================================================
# ASSIGN COLUMN NAMES
# ============================================================
cols = [
    "NUMBER","X_IMAGE","Y_IMAGE","RA","DEC",
    "MAG_ORIG","MAGERR_ORIG",
    "MAG_APER","MAGERR_APER",
    "FLAGS","ISOAREA_IMAGE",
    "MAG_DEGRADED","MAGERR_DEGRADED",
]

i_df.columns = cols
z_df.columns = cols
y_df.columns = cols

print(f"Y catalog size: {len(y_df)}")

# ============================================================
# USE DEGRADED MAGNITUDES
# ============================================================
for df in [i_df, z_df, y_df]:
    df["MAG_CAL"] = df["MAG_DEGRADED"]
    df["MAGERR_CAL"] = df["MAGERR_DEGRADED"]

# ============================================================
# MAGERR → SNR
# ============================================================
def magerr_to_snr(err):
    snr = 2.5 / (np.log(10) * err)
    snr[~np.isfinite(snr)] = 0.0
    return snr

# ============================================================
# Coordinate matching (Y reference)
# ============================================================
y_coords = SkyCoord(y_df["RA"].values, y_df["DEC"].values, unit="deg")
i_coords = SkyCoord(i_df["RA"].values, i_df["DEC"].values, unit="deg")
z_coords = SkyCoord(z_df["RA"].values, z_df["DEC"].values, unit="deg")

match_radius = 1.0 * u.arcsec

idx_i, sep_i, _ = y_coords.match_to_catalog_sky(i_coords)
idx_z, sep_z, _ = y_coords.match_to_catalog_sky(z_coords)

# ============================================================
# Matching function
# ============================================================
def matched_array(src_df, idx, sep):

    n = len(y_df)

    mag = np.full(n, np.nan)
    err = np.full(n, np.nan)

    detected = sep < match_radius

    mag[detected] = src_df["MAG_CAL"].values[idx[detected]]
    err[detected] = src_df["MAGERR_CAL"].values[idx[detected]]

    return mag, err, detected

# ============================================================
# Matched photometry
# ============================================================
i_mag, i_err, i_detected = matched_array(i_df, idx_i, sep_i)
z_mag, z_err, z_detected = matched_array(z_df, idx_z, sep_z)

y_mag = y_df["MAG_CAL"].values
y_err = y_df["MAGERR_CAL"].values
y_detected = np.ones(len(y_mag), dtype=bool)

# ============================================================
# Compute SNR
# ============================================================
i_snr = magerr_to_snr(i_err)
z_snr = magerr_to_snr(z_err)
y_snr = magerr_to_snr(y_err)

# ============================================================
# I-band nondetection logic
# ============================================================
I_LIMIT = 27.84

# nondetection if not detected OR very faint
i_nondet = (~i_detected) | (i_mag > I_LIMIT)

# assign limiting mag only for true nondetections
i_mag[~i_detected] = I_LIMIT
i_err[~i_detected] = 0.0

# ============================================================
# Colors
# ============================================================
color_zy = z_mag - y_mag
color_iz = i_mag - z_mag

color_zy_err = np.sqrt(z_err**2 + y_err**2)

# ============================================================
# Selection cuts
# ============================================================

cut_z_detected = z_detected
cut_z_snr = z_snr > 4.0
cut_y_snr = y_snr > 4.0

cut_zy = color_zy > 0.8
cut_break = color_iz > 1.0

# I-band dropout logic
cut_i_snr = i_snr < 2.0
cut_i_faint = i_mag > I_LIMIT

cut_i_dropout = (~i_detected) | cut_i_snr | cut_i_faint

# remove strong I detections
cut_no_strong_i = ~(i_detected & (i_snr > 3))

cut_sig = np.abs(color_zy) > 2.5 * color_zy_err

# ============================================================
# Final selection
# ============================================================

final_sel = (
    cut_z_detected &
    cut_z_snr &
    cut_y_snr &
    cut_zy &
    cut_break &
    cut_i_dropout &
    cut_no_strong_i &
    cut_sig
)
# ============================================================
# Build candidate catalog
# ============================================================
lbg_candidates = pd.DataFrame({

    "RA": y_df["RA"].values[lbg_idx],
    "DEC": y_df["DEC"].values[lbg_idx],

    "Y_mag": y_mag[lbg_idx],
    "Y_err": y_err[lbg_idx],

    "Z_mag": z_mag[lbg_idx],
    "Z_err": z_err[lbg_idx],
    "Z_SNR": z_snr[lbg_idx],

    "I_mag": i_mag[lbg_idx],
    "I_err": i_err[lbg_idx],
    "I_SNR": i_snr[lbg_idx],

    "Z-Y": color_zy[lbg_idx],
    "I-Z": color_iz[lbg_idx],

    "I_detected": i_detected[lbg_idx]
})

# ============================================================
# Remove red list sources
# ============================================================
red_df = pd.read_csv(redlist_file, sep=r"\s+", names=["RA","DEC"])

red_coords = SkyCoord(red_df["RA"].values, red_df["DEC"].values, unit="deg")

lbg_coords = SkyCoord(
    lbg_candidates["RA"].values,
    lbg_candidates["DEC"].values,
    unit="deg"
)

idx_red, sep_red, _ = lbg_coords.match_to_catalog_sky(red_coords)

lbg_candidates = lbg_candidates[sep_red.arcsec > 1.0]

# ============================================================
# Output catalog
# ============================================================
outpath = "/Users/aishwarya/Documents/Lyman_alpha/CDFS_candidates_ZY_degraded.cat"

lbg_candidates.to_csv(outpath, index=False)

print(f"Final candidates: {len(lbg_candidates)}")
print(f"Saved to {outpath}")

# ============================================================
# DS9 REGION FILES
# ============================================================

# circles
reg_arcsec_path = outpath.replace(".cat","_circles_1arcsec.reg")

with open(reg_arcsec_path,"w") as f:

    f.write("# Region file format: DS9 version 4.1\n")
    f.write("global color=red width=2\n")
    f.write("fk5\n")

    for _,row in lbg_candidates.iterrows():

        f.write(f'circle({row["RA"]},{row["DEC"]},1")\n')

print("Circle region file written")

# points
reg_point_path = outpath.replace(".cat","_points.reg")

with open(reg_point_path,"w") as f:

    f.write("# Region file format: DS9 version 4.1\n")
    f.write("global color=cyan point=cross 8 width=2\n")
    f.write("fk5\n")

    for _,row in lbg_candidates.iterrows():

        f.write(f'point({row["RA"]},{row["DEC"]})\n')

print("Point region file written")

i shape: (106333, 13)
z shape: (156831, 13)
y shape: (109373, 13)
Y catalog size: 109373
Final candidates: 355
Saved to /Users/aishwarya/Documents/Lyman_alpha/CDFS_candidates_ZY_degraded.cat
Circle region file written
Point region file written


In [34]:
import numpy as np
import pandas as pd
from astropy.coordinates import SkyCoord
import astropy.units as u

# ============================================================
# File paths
# ============================================================
i_file = "/Users/aishwarya/Desktop/new_cdfs/cat/LBG_I_degraded.cat"
z_file = "/Users/aishwarya/Desktop/new_cdfs/cat/LBG_Z_degraded.cat"
y_file = "/Users/aishwarya/Desktop/new_cdfs/cat/LBG_Y_degraded.cat"
redlist_file = "/Users/aishwarya/Downloads/candidates_red_list_cdfs.txt"

# ============================================================
# READ FILES
# ============================================================
i_df = pd.read_csv(i_file, sep=r"\s+", comment="#", header=None)
z_df = pd.read_csv(z_file, sep=r"\s+", comment="#", header=None)
y_df = pd.read_csv(y_file, sep=r"\s+", comment="#", header=None)

print("i shape:", i_df.shape)
print("z shape:", z_df.shape)
print("y shape:", y_df.shape)

# ============================================================
# ASSIGN COLUMN NAMES
# ============================================================
cols = [
    "NUMBER","X_IMAGE","Y_IMAGE","RA","DEC",
    "MAG_ORIG","MAGERR_ORIG",
    "MAG_APER","MAGERR_APER",
    "FLAGS","ISOAREA_IMAGE",
    "MAG_DEGRADED","MAGERR_DEGRADED",
]

i_df.columns = cols
z_df.columns = cols
y_df.columns = cols

print(f"Y catalog size: {len(y_df)}")

# ============================================================
# USE DEGRADED MAGNITUDES
# ============================================================
for df in [i_df, z_df, y_df]:
    df["MAG_CAL"] = df["MAG_DEGRADED"]
    df["MAGERR_CAL"] = df["MAGERR_DEGRADED"]

# ============================================================
# MAGERR → SNR
# ============================================================
def magerr_to_snr(err):
    snr = 2.5 / (np.log(10) * err)
    snr[~np.isfinite(snr)] = 0.0
    return snr

# ============================================================
# Coordinate matching (Y reference)
# ============================================================
y_coords = SkyCoord(y_df["RA"].values, y_df["DEC"].values, unit="deg")
i_coords = SkyCoord(i_df["RA"].values, i_df["DEC"].values, unit="deg")
z_coords = SkyCoord(z_df["RA"].values, z_df["DEC"].values, unit="deg")

match_radius = 1.0 * u.arcsec

idx_i, sep_i, _ = y_coords.match_to_catalog_sky(i_coords)
idx_z, sep_z, _ = y_coords.match_to_catalog_sky(z_coords)

# ============================================================
# Matching function
# ============================================================
def matched_array(src_df, idx, sep):

    n = len(y_df)

    mag = np.full(n, np.nan)
    err = np.full(n, np.nan)

    detected = sep < match_radius

    mag[detected] = src_df["MAG_CAL"].values[idx[detected]]
    err[detected] = src_df["MAGERR_CAL"].values[idx[detected]]

    return mag, err, detected

# ============================================================
# Matched photometry
# ============================================================
i_mag, i_err, i_detected = matched_array(i_df, idx_i, sep_i)
z_mag, z_err, z_detected = matched_array(z_df, idx_z, sep_z)

y_mag = y_df["MAG_CAL"].values
y_err = y_df["MAGERR_CAL"].values

# ============================================================
# Compute SNR
# ============================================================
i_snr = magerr_to_snr(i_err)
z_snr = magerr_to_snr(z_err)
y_snr = magerr_to_snr(y_err)

# ============================================================
# I-band nondetection handling
# ============================================================
I_LIMIT = 27.84

# Replace missing I-band detections with limiting magnitude
i_mag[~i_detected] = I_LIMIT
i_err[~i_detected] = np.inf

# ============================================================
# Colors
# ============================================================
color_zy = z_mag - y_mag
color_iz = i_mag - z_mag

color_zy_err = np.sqrt(z_err**2 + y_err**2)

# ============================================================
# Selection cuts
# ============================================================

cut_z_detected = z_detected
cut_z_snr = z_snr > 3.0

# Require strong Y detection (reduces noise matches)
cut_y_snr = y_snr > 3.0

# color cuts
cut_zy = color_zy > 0.8
cut_break = color_iz > 1.0

# I-band dropout condition
cut_i_dropout = (i_snr < 2.0) | (i_mag > I_LIMIT)

# color significance
cut_sig = color_zy > 2.5 * color_zy_err
cut_area = y_df["ISOAREA_IMAGE"].values < 40

# ============================================================
# Final selection
# ============================================================
final_sel = (
    cut_z_detected &
    cut_z_snr &
    cut_y_snr &
    cut_zy &
    cut_break &
    cut_i_dropout &
    cut_sig &
    cut_area
)

# ============================================================
# Candidate indices
# ============================================================
lbg_idx = np.where(final_sel)[0]

print("Selected candidates:", len(lbg_idx))

# ============================================================
# Build candidate catalog
# ============================================================
lbg_candidates = pd.DataFrame({

    "RA": y_df["RA"].values[lbg_idx],
    "DEC": y_df["DEC"].values[lbg_idx],

    "Y_mag": y_mag[lbg_idx],
    "Y_err": y_err[lbg_idx],

    "Z_mag": z_mag[lbg_idx],
    "Z_err": z_err[lbg_idx],
    "Z_SNR": z_snr[lbg_idx],

    "I_mag": i_mag[lbg_idx],
    "I_err": i_err[lbg_idx],
    "I_SNR": i_snr[lbg_idx],

    "Z-Y": color_zy[lbg_idx],
    "I-Z": color_iz[lbg_idx]
})

# ============================================================
# Remove red list sources
# ============================================================
red_df = pd.read_csv(redlist_file, sep=r"\s+", names=["RA","DEC"])

red_coords = SkyCoord(red_df["RA"].values, red_df["DEC"].values, unit="deg")

lbg_coords = SkyCoord(
    lbg_candidates["RA"].values,
    lbg_candidates["DEC"].values,
    unit="deg"
)

idx_red, sep_red, _ = lbg_coords.match_to_catalog_sky(red_coords)

lbg_candidates = lbg_candidates[sep_red.arcsec > 1.0]

# ============================================================
# Output catalog
# ============================================================
outpath = "/Users/aishwarya/Desktop/CDFS_candidates_ZY_degraded.cat"

lbg_candidates.to_csv(outpath, index=False)

print(f"Final candidates: {len(lbg_candidates)}")
print(f"Saved to {outpath}")

# ============================================================
# DS9 REGION FILES
# ============================================================

reg_arcsec_path = outpath.replace(".cat","_circles_1arcsec.reg")

with open(reg_arcsec_path,"w") as f:

    f.write("# Region file format: DS9 version 4.1\n")
    f.write("global color=red width=2\n")
    f.write("fk5\n")

    for _,row in lbg_candidates.iterrows():
        f.write(f'circle({row["RA"]},{row["DEC"]},1")\n')

print("Circle region file written")

reg_point_path = outpath.replace(".cat","_points.reg")

with open(reg_point_path,"w") as f:

    f.write("# Region file format: DS9 version 4.1\n")
    f.write("global color=cyan point=cross 8 width=2\n")
    f.write("fk5\n")

    for _,row in lbg_candidates.iterrows():
        f.write(f'point({row["RA"]},{row["DEC"]})\n')

print(f"Point region file written {reg_point_path}")

i shape: (106333, 13)
z shape: (156831, 13)
y shape: (109373, 13)
Y catalog size: 109373
Selected candidates: 433
Final candidates: 43
Saved to /Users/aishwarya/Desktop/CDFS_candidates_ZY_degraded.cat
Circle region file written
Point region file written /Users/aishwarya/Desktop/CDFS_candidates_ZY_degraded_points.reg


In [27]:
from astropy.coordinates import SkyCoord
import astropy.units as u

# ============================================================
# FILES
# ============================================================
region_file = "/Users/aishwarya/Desktop/source_cdfs.reg"
amb_file = "/Users/aishwarya/Desktop/amb_cdfs.txt"

# ============================================================
# READ REGION FILE (extract RA, DEC)
# ============================================================
reg_ra, reg_dec = [], []

with open(region_file, "r") as f:
    for line in f:
        line = line.strip()

        if line.startswith("point") or line.startswith("circle"):
            coords = line.split("(")[1].split(")")[0].split(",")
            reg_ra.append(float(coords[0]))
            reg_dec.append(float(coords[1]))

reg_coords = SkyCoord(reg_ra, reg_dec, unit="deg")

# ============================================================
# READ AMBIGUOUS SOURCES
# ============================================================
amb_df = pd.read_csv(amb_file, sep=r"\s+", names=["RA","DEC"])
amb_coords = SkyCoord(amb_df["RA"].values, amb_df["DEC"].values, unit="deg")

# ============================================================
# MATCH CANDIDATES TO REGION (KEEP ONLY THESE)
# ============================================================
cand_coords = SkyCoord(
    lbg_candidates["RA"].values,
    lbg_candidates["DEC"].values,
    unit="deg"
)

idx_reg, sep_reg, _ = cand_coords.match_to_catalog_sky(reg_coords)

keep_region = sep_reg < 1.0 * u.arcsec

clean_candidates = lbg_candidates[keep_region].copy()

print("After region filter:", len(clean_candidates))

# ============================================================
# REMOVE AMBIGUOUS SOURCES
# ============================================================
cand_coords_clean = SkyCoord(
    clean_candidates["RA"].values,
    clean_candidates["DEC"].values,
    unit="deg"
)

idx_amb, sep_amb, _ = cand_coords_clean.match_to_catalog_sky(amb_coords)

keep_clean = sep_amb > 1.0 * u.arcsec

clean_candidates = clean_candidates[keep_clean]

print("After removing ambiguous:", len(clean_candidates))

# ============================================================
# SAVE CLEAN CATALOG
# ============================================================
clean_out = outpath.replace(".cat", "_CLEAN.cat")
clean_candidates.to_csv(clean_out, index=False)

print("Saved clean catalog:", clean_out)

# ============================================================
# CLEAN REGION FILES
# ============================================================

# --- 1 arcsec circles ---
reg_arcsec_clean = clean_out.replace(".cat", "_circles_1arcsec.reg")

with open(reg_arcsec_clean, "w") as f:
    f.write("# Region file format: DS9 version 4.1\n")
    f.write("global color=green width=2\n")
    f.write("fk5\n")

    for _, row in clean_candidates.iterrows():
        f.write(f'circle({row["RA"]},{row["DEC"]},1")\n')

print("Clean 1 arcsec region file saved")

# --- point (cross) ---
reg_point_clean = clean_out.replace(".cat", "_points.reg")

with open(reg_point_clean, "w") as f:
    f.write("# Region file format: DS9 version 4.1\n")
    f.write("global color=yellow point=cross 8 width=2\n")
    f.write("fk5\n")

    for _, row in clean_candidates.iterrows():
        f.write(f'point({row["RA"]},{row["DEC"]})\n')

print("Clean point region file saved")

ValueError: Latitude angle(s) must be within -90 deg <= angle <= 90 deg, got 5901.5533 deg

In [11]:
print(y_snr[lbg_idx])

[12.17801599  8.30643737  5.28603027 ...  9.42054591 11.91212951
  7.43260326]


In [5]:
print("I-band match separations (arcsec):")
print(np.percentile(sep_i.arcsec, [50, 90, 95, 99]))

I-band match separations (arcsec):
[ 0.40940817  7.85331523 10.10889221 14.50907107]


In [6]:
i_ra = i_df["RA"].values[idx_i[lbg_idx]]
i_dec = i_df["DEC"].values[idx_i[lbg_idx]]

print((i_ra - y_df["RA"].values[lbg_idx]) * 3600)
print((i_dec - y_df["DEC"].values[lbg_idx]) * 3600)

[-0.31428 -0.45252  5.6412  ...  2.64564 -2.74788  5.42124]
[0.28332 1.17108 0.50616 ... 0.26712 2.99268 0.03744]


In [28]:
import numpy as np
import pandas as pd
from astropy.coordinates import SkyCoord
import astropy.units as u

# ============================================================
# File paths
# ============================================================
i_file = "/Users/aishwarya/Desktop/Lyman_alpha_2/CAT/i_band_degraded.cat"
z_file = "/Users/aishwarya/Desktop/Lyman_alpha_2/CAT/z_band_degraded.cat"
y_file = "/Users/aishwarya/Desktop/new_cdfs/cat/LBG_Y_degraded.cat"
redlist_file = "/Users/aishwarya/Downloads/candidates_red_list_cdfs.txt"

# ============================================================
# READ FILES
# ============================================================
i_df = pd.read_csv(i_file, sep=r"\s+", comment="#", header=None)
z_df = pd.read_csv(z_file, sep=r"\s+", comment="#", header=None)
y_df = pd.read_csv(y_file, sep=r"\s+", comment="#", header=None)

print("i shape:", i_df.shape)
print("z shape:", z_df.shape)
print("y shape:", y_df.shape)

# ============================================================
# COLUMN NAMES
# ============================================================

cols = [
    "NUMBER","X_IMAGE","Y_IMAGE","RA","DEC",
    "MAG_ORIG","MAGERR_ORIG",
    "MAG_APER","MAGERR_APER",
    "FLAGS",
    "MAG_DEGRADED","MAGERR_DEGRADED"
]

i_df.columns = cols
z_df.columns = cols
y_df.columns = cols

print(f"Y catalog size: {len(y_df)}")

# ============================================================
# USE DEGRADED MAGNITUDES
# ============================================================

for df in [i_df,z_df,y_df]:
    df["MAG_CAL"] = df["MAG_DEGRADED"]
    df["MAGERR_CAL"] = df["MAGERR_DEGRADED"]

# ============================================================
# SNR from magnitude error
# ============================================================

def magerr_to_snr(err):

    snr = 2.5 / (np.log(10) * err)

    snr[~np.isfinite(snr)] = 0

    return snr

# ============================================================
# COORDINATES
# ============================================================

y_coords = SkyCoord(y_df["RA"].values, y_df["DEC"].values, unit="deg")
i_coords = SkyCoord(i_df["RA"].values, i_df["DEC"].values, unit="deg")
z_coords = SkyCoord(z_df["RA"].values, z_df["DEC"].values, unit="deg")

match_radius = 3 * u.arcsec

# ============================================================
# MATCH CATALOGS
# ============================================================

idx_i, sep_i, _ = y_coords.match_to_catalog_sky(i_coords)
idx_z, sep_z, _ = y_coords.match_to_catalog_sky(z_coords)

def matched_array(src_df, idx, sep):

    n = len(y_df)

    mag = np.full(n, np.nan)
    err = np.full(n, np.nan)

    detected = sep < match_radius

    mag[detected] = src_df["MAG_CAL"].values[idx[detected]]
    err[detected] = src_df["MAGERR_CAL"].values[idx[detected]]

    return mag, err, detected

# ============================================================
# MATCHED PHOTOMETRY
# ============================================================

i_mag, i_err, i_detected = matched_array(i_df, idx_i, sep_i)
z_mag, z_err, z_detected = matched_array(z_df, idx_z, sep_z)

# Y photometry comes directly
y_mag = y_df["MAG_CAL"].values
y_err = y_df["MAGERR_CAL"].values
y_detected = np.ones(len(y_df),dtype=bool)

# ============================================================
# COMPUTE SNR
# ============================================================

i_snr = magerr_to_snr(i_err)
z_snr = magerr_to_snr(z_err)
y_snr = magerr_to_snr(y_err)

# ============================================================
# STRICT I-DROPOUT CONDITION
# ============================================================

I_LIMIT = 27.84

i_nondet = i_snr < 2.0

i_mag[i_nondet] = I_LIMIT
i_err[i_nondet] = np.nan

# ============================================================
# COLORS
# ============================================================

color_zy = z_mag - y_mag
color_iz = i_mag - z_mag

color_zy_err = np.sqrt(z_err**2 + y_err**2)

# ============================================================
# SELECTION CUTS
# ============================================================

cut_z_detected = z_detected
cut_z_snr = z_snr > 5
cut_y_snr = y_snr > 5

cut_zy = color_zy > 0.8
cut_break = color_iz > 1.0

cut_i_dropout = i_nondet

cut_sig = color_zy > 2.5 * color_zy_err

# ============================================================
# FINAL SELECTION
# ============================================================

final_sel = (
    cut_z_detected &
    cut_z_snr &
    cut_y_snr &
    cut_zy &
    cut_break &
    cut_i_dropout &
    cut_sig
)

lbg_idx = np.where(final_sel)[0]

print(f"After SNR cuts: {len(lbg_idx)} candidates")

# ============================================================
# BUILD CANDIDATE CATALOG
# ============================================================

lbg_candidates = pd.DataFrame({

"RA": y_df["RA"].values[lbg_idx],
"DEC": y_df["DEC"].values[lbg_idx],

"Y_mag": y_mag[lbg_idx],
"Y_err": y_err[lbg_idx],

"Z_mag": z_mag[lbg_idx],
"Z_err": z_err[lbg_idx],
"Z_SNR": z_snr[lbg_idx],

"I_mag": i_mag[lbg_idx],
"I_err": i_err[lbg_idx],
"I_SNR": i_snr[lbg_idx],

"Z-Y": color_zy[lbg_idx],
"I-Z": color_iz[lbg_idx],

"I_detected": i_detected[lbg_idx]

})

# ============================================================
# REMOVE RED LIST SOURCES
# ============================================================

red_df = pd.read_csv(redlist_file,sep=r"\s+",names=["RA","DEC"])

red_coords = SkyCoord(red_df["RA"].values,red_df["DEC"].values,unit="deg")

lbg_coords = SkyCoord(lbg_candidates["RA"].values,
                      lbg_candidates["DEC"].values,
                      unit="deg")

idx_red, sep_red, _ = lbg_coords.match_to_catalog_sky(red_coords)

lbg_candidates = lbg_candidates[sep_red.arcsec > 1.0]

# ============================================================
# SAVE OUTPUT
# ============================================================

outpath = "/Users/aishwarya/Desktop/CDFS_candidates_ZY_degraded.cat"

lbg_candidates.to_csv(outpath,index=False)

print(f"Final candidates: {len(lbg_candidates)}")
print(f"Saved to {outpath}")

# ============================================================
# DS9 REGION FILES
# ============================================================

reg_arcsec_path = outpath.replace(".cat","_circles_1arcsec.reg")

with open(reg_arcsec_path,"w") as f:

    f.write("# Region file format: DS9 version 4.1\n")
    f.write("global color=red width=2\n")
    f.write("fk5\n")

    for _,row in lbg_candidates.iterrows():

        f.write(f'circle({row["RA"]},{row["DEC"]},1")\n')

print("Circle region file saved")

reg_point_path = outpath.replace(".cat","_points.reg")

with open(reg_point_path,"w") as f:

    f.write("# Region file format: DS9 version 4.1\n")
    f.write("global color=cyan point=cross 8 width=2\n")
    f.write("fk5\n")

    for _,row in lbg_candidates.iterrows():

        f.write(f'point({row["RA"]},{row["DEC"]})\n')

print("Point region file saved")

i shape: (239300, 12)
z shape: (297003, 12)
y shape: (286091, 12)
Y catalog size: 286091
After SNR cuts: 3753 candidates
Final candidates: 3674
Saved to /Users/aishwarya/Desktop/CDFS_candidates_ZY_degraded.cat
Circle region file saved
Point region file saved


In [40]:
import pandas as pd
import numpy as np
from astropy.coordinates import SkyCoord
import astropy.units as u

# ============================================================
# FILE PATHS
# ============================================================
file_check = "/Users/aishwarya/Desktop/source_cdfs_clean.cat"
file_ref   = "/Users/aishwarya/Desktop/CDFS_candidates_ZY_degraded.cat"

# ============================================================
# LOAD FILES
# ============================================================
df_check = pd.read_csv(file_check, sep=r"\s+")
df_ref   = pd.read_csv(file_ref,   sep=",")

# clean column names
df_check.columns = df_check.columns.str.strip()
df_ref.columns   = df_ref.columns.str.strip()

# ============================================================
# EXTRACT RA/DEC
# ============================================================
ra_check  = df_check["ALPHA_J2000"].astype(float).values
dec_check = df_check["DELTA_J2000"].astype(float).values

ra_ref  = df_ref["RA"].astype(float).values
dec_ref = df_ref["DEC"].astype(float).values

# ============================================================
# SKYCOORD
# ============================================================
coords_check = SkyCoord(ra=ra_check*u.deg, dec=dec_check*u.deg)
coords_ref   = SkyCoord(ra=ra_ref*u.deg,   dec=dec_ref*u.deg)

# ============================================================
# MATCHING
# ============================================================
idx, d2d, _ = coords_check.match_to_catalog_sky(coords_ref)

tolerance = 1.0 * u.arcsec
matched = d2d < tolerance

# ============================================================
# RESULTS SUMMARY
# ============================================================
print(f"Total sources: {len(coords_check)}")
print(f"Matched: {np.sum(matched)}")
print(f"Missing: {np.sum(~matched)}")

# ============================================================
# 🔍 PRINT MISSING SOURCES
# ============================================================
print("\nMissing sources (RA, DEC, nearest separation in arcsec):")

for i in np.where(~matched)[0]:
    ra  = ra_check[i]
    dec = dec_check[i]
    sep = d2d[i].arcsec

    print(f"RA: {ra:.6f}, DEC: {dec:.6f}, nearest_sep: {sep:.3f} arcsec")

# ============================================================
# SAVE MISSING SOURCES WITH INFO
# ============================================================
missing_df = df_check.loc[~matched].copy()
missing_df["nearest_sep_arcsec"] = d2d[~matched].arcsec

missing_df.to_csv(
    "/Users/aishwarya/Desktop/missing_sources_detailed.cat",
    sep=' ',
    index=False
)

print("\nSaved detailed missing sources → missing_sources_detailed.cat")

Total sources: 19
Matched: 18
Missing: 1

Missing sources (RA, DEC, nearest separation in arcsec):
RA: 53.235486, DEC: -27.515785, nearest_sep: 647.816 arcsec

Saved detailed missing sources → missing_sources_detailed.cat


In [29]:
import numpy as np
import pandas as pd
from astropy.coordinates import SkyCoord
import astropy.units as u

# ============================================================
# File paths
# ============================================================

i_file = "/Users/aishwarya/Desktop/Lyman_alpha_2/CAT/i_band_degraded.cat"
z_file = "/Users/aishwarya/Desktop/Lyman_alpha_2/CAT/z_band_degraded.cat"
y_file = "/Users/aishwarya/Desktop/new_cdfs/cat/LBG_Y_degraded.cat"
redlist_file = "/Users/aishwarya/Downloads/candidates_red_list_cdfs.txt"

# ============================================================
# READ FILES
# ============================================================

i_df = pd.read_csv(i_file, sep=r"\s+", comment="#", header=None)
z_df = pd.read_csv(z_file, sep=r"\s+", comment="#", header=None)
y_df = pd.read_csv(y_file, sep=r"\s+", comment="#", header=None)

print("i shape:", i_df.shape)
print("z shape:", z_df.shape)
print("y shape:", y_df.shape)

# ============================================================
# COLUMN NAMES
# ============================================================

cols = [
    "NUMBER","X_IMAGE","Y_IMAGE","RA","DEC",
    "MAG_ORIG","MAGERR_ORIG",
    "MAG_APER","MAGERR_APER",
    "FLAGS",
    "MAG_DEGRADED","MAGERR_DEGRADED"
]

i_df.columns = cols
z_df.columns = cols
y_df.columns = cols

print(f"Y catalog size (before FLAG cleaning): {len(y_df)}")

# ============================================================
# REMOVE BAD SExtractor FLAGS
# ============================================================

flag_limit = 3

i_df = i_df[i_df["FLAGS"] <= flag_limit].reset_index(drop=True)
z_df = z_df[z_df["FLAGS"] <= flag_limit].reset_index(drop=True)
y_df = y_df[y_df["FLAGS"] <= flag_limit].reset_index(drop=True)

print("After FLAG filtering:")
print("i catalog:", len(i_df))
print("z catalog:", len(z_df))
print("y catalog:", len(y_df))

# ============================================================
# USE DEGRADED MAGNITUDES
# ============================================================

for df in [i_df, z_df, y_df]:
    df["MAG_CAL"] = df["MAG_DEGRADED"]
    df["MAGERR_CAL"] = df["MAGERR_DEGRADED"]

# ============================================================
# SNR from magnitude error
# ============================================================

def magerr_to_snr(err):

    snr = 2.5 / (np.log(10) * err)

    snr[~np.isfinite(snr)] = 0

    return snr

# ============================================================
# COORDINATES
# ============================================================

y_coords = SkyCoord(y_df["RA"].values, y_df["DEC"].values, unit="deg")
i_coords = SkyCoord(i_df["RA"].values, i_df["DEC"].values, unit="deg")
z_coords = SkyCoord(z_df["RA"].values, z_df["DEC"].values, unit="deg")

match_radius = 3 * u.arcsec

# ============================================================
# MATCH CATALOGS
# ============================================================

idx_i, sep_i, _ = y_coords.match_to_catalog_sky(i_coords)
idx_z, sep_z, _ = y_coords.match_to_catalog_sky(z_coords)

def matched_array(src_df, idx, sep):

    n = len(y_df)

    mag = np.full(n, np.nan)
    err = np.full(n, np.nan)

    detected = sep < match_radius

    mag[detected] = src_df["MAG_CAL"].values[idx[detected]]
    err[detected] = src_df["MAGERR_CAL"].values[idx[detected]]

    return mag, err, detected

# ============================================================
# MATCHED PHOTOMETRY
# ============================================================

i_mag, i_err, i_detected = matched_array(i_df, idx_i, sep_i)
z_mag, z_err, z_detected = matched_array(z_df, idx_z, sep_z)

# Y photometry comes directly
y_mag = y_df["MAG_CAL"].values
y_err = y_df["MAGERR_CAL"].values
y_detected = np.ones(len(y_df), dtype=bool)

# ============================================================
# COMPUTE SNR
# ============================================================

i_snr = magerr_to_snr(i_err)
z_snr = magerr_to_snr(z_err)
y_snr = magerr_to_snr(y_err)

# ============================================================
# STRICT I-DROPOUT CONDITION
# ============================================================

I_LIMIT = 27.84

i_nondet = i_snr < 2.0

i_mag[i_nondet] = I_LIMIT
i_err[i_nondet] = np.nan

# ============================================================
# COLORS
# ============================================================

color_zy = z_mag - y_mag
color_iz = i_mag - z_mag

color_zy_err = np.sqrt(z_err**2 + y_err**2)

# ============================================================
# SELECTION CUTS
# ============================================================

cut_z_detected = z_detected
cut_z_snr = z_snr > 5
cut_y_snr = y_snr > 5

cut_zy = color_zy > 0.8
cut_break = color_iz > 1.0

cut_i_dropout = i_nondet

cut_sig = color_zy > 2.5 * color_zy_err

# ============================================================
# FINAL SELECTION
# ============================================================

final_sel = (
    cut_z_detected &
    cut_z_snr &
    cut_y_snr &
    cut_zy &
    cut_break &
    cut_i_dropout &
    cut_sig
)

lbg_idx = np.where(final_sel)[0]

print(f"After SNR cuts: {len(lbg_idx)} candidates")

# ============================================================
# BUILD CANDIDATE CATALOG
# ============================================================

lbg_candidates = pd.DataFrame({

"RA": y_df["RA"].values[lbg_idx],
"DEC": y_df["DEC"].values[lbg_idx],

"Y_mag": y_mag[lbg_idx],
"Y_err": y_err[lbg_idx],

"Z_mag": z_mag[lbg_idx],
"Z_err": z_err[lbg_idx],
"Z_SNR": z_snr[lbg_idx],

"I_mag": i_mag[lbg_idx],
"I_err": i_err[lbg_idx],
"I_SNR": i_snr[lbg_idx],

"Z-Y": color_zy[lbg_idx],
"I-Z": color_iz[lbg_idx],

"I_detected": i_detected[lbg_idx]

})

# ============================================================
# REMOVE RED LIST SOURCES
# ============================================================

red_df = pd.read_csv(redlist_file, sep=r"\s+", names=["RA","DEC"])

red_coords = SkyCoord(red_df["RA"].values, red_df["DEC"].values, unit="deg")

lbg_coords = SkyCoord(
    lbg_candidates["RA"].values,
    lbg_candidates["DEC"].values,
    unit="deg"
)

idx_red, sep_red, _ = lbg_coords.match_to_catalog_sky(red_coords)

lbg_candidates = lbg_candidates[sep_red.arcsec > 1.0]

# ============================================================
# SAVE OUTPUT
# ============================================================

outpath = "/Users/aishwarya/Desktop/CDFS_candidates_ZY_degraded.cat"

lbg_candidates.to_csv(outpath, index=False)

print(f"Final candidates: {len(lbg_candidates)}")
print(f"Saved to {outpath}")

# ============================================================
# DS9 REGION FILES
# ============================================================

reg_arcsec_path = outpath.replace(".cat","_circles_1arcsec.reg")

with open(reg_arcsec_path,"w") as f:

    f.write("# Region file format: DS9 version 4.1\n")
    f.write("global color=red width=2\n")
    f.write("fk5\n")

    for _,row in lbg_candidates.iterrows():

        f.write(f'circle({row["RA"]},{row["DEC"]},1")\n')

print("Circle region file saved")

reg_point_path = outpath.replace(".cat","_points.reg")

with open(reg_point_path,"w") as f:

    f.write("# Region file format: DS9 version 4.1\n")
    f.write("global color=cyan point=cross 8 width=2\n")
    f.write("fk5\n")

    for _,row in lbg_candidates.iterrows():

        f.write(f'point({row["RA"]},{row["DEC"]})\n')

print("Point region file saved")

i shape: (239300, 12)
z shape: (297003, 12)
y shape: (286091, 12)
Y catalog size (before FLAG cleaning): 286091
After FLAG filtering:
i catalog: 239199
z catalog: 296872
y catalog: 286008
After SNR cuts: 3732 candidates
Final candidates: 3653
Saved to /Users/aishwarya/Desktop/CDFS_candidates_ZY_degraded.cat
Circle region file saved
Point region file saved


In [37]:
import numpy as np
import pandas as pd
from astropy.coordinates import SkyCoord
import astropy.units as u

# ============================================================
# File paths
# ============================================================

i_file = "/Users/aishwarya/Desktop/Lyman_alpha_2/CAT/i_band_degraded.cat"
z_file = "/Users/aishwarya/Desktop/Lyman_alpha_2/CAT/z_band_degraded.cat"
y_file = "/Users/aishwarya/Desktop/new_cdfs/cat/LBG_Y_degraded.cat"
redlist_file = "/Users/aishwarya/Downloads/candidates_red_list_cdfs.txt"

# ============================================================
# READ FILES
# ============================================================

i_df = pd.read_csv(i_file, sep=r"\s+", comment="#", header=None)
z_df = pd.read_csv(z_file, sep=r"\s+", comment="#", header=None)
y_df = pd.read_csv(y_file, sep=r"\s+", comment="#", header=None)

print("i shape:", i_df.shape)
print("z shape:", z_df.shape)
print("y shape:", y_df.shape)

# ============================================================
# COLUMN NAMES
# ============================================================

cols = [
"NUMBER","X_IMAGE","Y_IMAGE","RA","DEC",
"MAG_ORIG","MAGERR_ORIG",
"MAG_APER","MAGERR_APER",
"FLAGS",
"MAG_DEGRADED","MAGERR_DEGRADED"
]

i_df.columns = cols
z_df.columns = cols
y_df.columns = cols

print(f"Y catalog size (before FLAG cleaning): {len(y_df)}")

# ============================================================
# REMOVE BAD FLAGS
# ============================================================

flag_limit = 3

i_df = i_df[i_df["FLAGS"] <= flag_limit].reset_index(drop=True)
z_df = z_df[z_df["FLAGS"] <= flag_limit].reset_index(drop=True)
y_df = y_df[y_df["FLAGS"] <= flag_limit].reset_index(drop=True)

print("After FLAG filtering:")
print("i catalog:", len(i_df))
print("z catalog:", len(z_df))
print("y catalog:", len(y_df))

# ============================================================
# USE DEGRADED MAGNITUDES
# ============================================================

for df in [i_df, z_df, y_df]:
    df["MAG_CAL"] = df["MAG_DEGRADED"]
    df["MAGERR_CAL"] = df["MAGERR_DEGRADED"]

# ============================================================
# SNR from magnitude error
# ============================================================

def magerr_to_snr(err):

    snr = 2.5 / (np.log(10) * err)
    snr[~np.isfinite(snr)] = 0

    return snr

# ============================================================
# COORDINATES
# ============================================================

y_coords = SkyCoord(y_df["RA"].values, y_df["DEC"].values, unit="deg")
i_coords = SkyCoord(i_df["RA"].values, i_df["DEC"].values, unit="deg")
z_coords = SkyCoord(z_df["RA"].values, z_df["DEC"].values, unit="deg")

match_radius = 3 * u.arcsec

# ============================================================
# MATCH CATALOGS
# ============================================================

idx_i, sep_i, _ = y_coords.match_to_catalog_sky(i_coords)
idx_z, sep_z, _ = y_coords.match_to_catalog_sky(z_coords)

def matched_array(src_df, idx, sep):

    n = len(y_df)

    mag = np.full(n, np.nan)
    err = np.full(n, np.nan)

    detected = sep < match_radius

    mag[detected] = src_df["MAG_CAL"].values[idx[detected]]
    err[detected] = src_df["MAGERR_CAL"].values[idx[detected]]

    return mag, err, detected

# ============================================================
# MATCHED PHOTOMETRY
# ============================================================

i_mag, i_err, i_detected = matched_array(i_df, idx_i, sep_i)
z_mag, z_err, z_detected = matched_array(z_df, idx_z, sep_z)

y_mag = y_df["MAG_CAL"].values
y_err = y_df["MAGERR_CAL"].values
y_detected = np.ones(len(y_df), dtype=bool)

# ============================================================
# COMPUTE SNR
# ============================================================

i_snr = magerr_to_snr(i_err)
z_snr = magerr_to_snr(z_err)
y_snr = magerr_to_snr(y_err)

# ============================================================
# STRICT I-DROPOUT CONDITION
# ============================================================

I_LIMIT = 27.84

i_nondet = i_snr < 2.0

i_mag[i_nondet] = I_LIMIT
i_err[i_nondet] = np.nan

# ============================================================
# COLORS
# ============================================================

color_zy = z_mag - y_mag
color_iz = i_mag - z_mag

color_zy_err = np.sqrt(z_err**2 + y_err**2)

# ============================================================
# SELECTION CUTS
# ============================================================

cut_z_detected = z_detected
cut_z_snr = z_snr > 5
cut_y_snr = y_snr > 5

cut_zy = color_zy > 0.8
cut_break = color_iz > 1.0

cut_i_dropout = i_nondet

cut_sig = color_zy > 2.5 * color_zy_err

# ============================================================
# FINAL SELECTION
# ============================================================

final_sel = (
cut_z_detected &
cut_z_snr &
cut_y_snr &
cut_zy &
cut_break &
cut_i_dropout &
cut_sig
)

lbg_idx = np.where(final_sel)[0]

print(f"After SNR cuts: {len(lbg_idx)} candidates")

# ============================================================
# BUILD CANDIDATE CATALOG
# ============================================================

lbg_candidates = pd.DataFrame({

"RA": y_df["RA"].values[lbg_idx],
"DEC": y_df["DEC"].values[lbg_idx],

"Y_mag": y_mag[lbg_idx],
"Y_err": y_err[lbg_idx],

"Z_mag": z_mag[lbg_idx],
"Z_err": z_err[lbg_idx],
"Z_SNR": z_snr[lbg_idx],

"I_mag": i_mag[lbg_idx],
"I_err": i_err[lbg_idx],
"I_SNR": i_snr[lbg_idx],

"Z-Y": color_zy[lbg_idx],
"I-Z": color_iz[lbg_idx],

"I_detected": i_detected[lbg_idx]

})

# ============================================================
# REMOVE SOURCES NEAR BRIGHT STARS
# ============================================================

bright_star_mag = 20

bright_stars = bright_stars = pd.concat([
    i_df[i_df["MAG_CAL"] < 20],
    z_df[z_df["MAG_CAL"] < 20]
])
star_coords = SkyCoord(
bright_stars["RA"].values,
bright_stars["DEC"].values,
unit="deg"
)

cand_coords = SkyCoord(
lbg_candidates["RA"].values,
lbg_candidates["DEC"].values,
unit="deg"
)

idx_cand, idx_star, sep, _ = cand_coords.search_around_sky(
    star_coords,
    25 * u.arcsec
)

remove_idx = np.unique(idx_cand)

mask = np.ones(len(lbg_candidates), dtype=bool)
mask[remove_idx] = False

lbg_candidates = lbg_candidates[mask].reset_index(drop=True)

print("Removed near bright stars:", len(remove_idx))
# ============================================================
# REMOVE RED LIST SOURCES
# ============================================================

red_df = pd.read_csv(redlist_file, sep=r"\s+", names=["RA","DEC"])

red_coords = SkyCoord(red_df["RA"].values, red_df["DEC"].values, unit="deg")

lbg_coords = SkyCoord(
lbg_candidates["RA"].values,
lbg_candidates["DEC"].values,
unit="deg"
)

idx_red, sep_red, _ = lbg_coords.match_to_catalog_sky(red_coords)

lbg_candidates = lbg_candidates[sep_red.arcsec > 1.0]

# ============================================================
# SAVE OUTPUT
# ============================================================

outpath = "/Users/aishwarya/Desktop/CDFS_candidates_ZY_degraded.cat"

lbg_candidates.to_csv(outpath, index=False)

print(f"Final candidates: {len(lbg_candidates)}")
print(f"Saved to {outpath}")

# ============================================================
# DS9 REGION FILES
# ============================================================

reg_arcsec_path = outpath.replace(".cat","_circles_1arcsec.reg")

with open(reg_arcsec_path,"w") as f:

    f.write("# Region file format: DS9 version 4.1\n")
    f.write("global color=red width=2\n")
    f.write("fk5\n")

    for _,row in lbg_candidates.iterrows():

        f.write(f'circle({row["RA"]},{row["DEC"]},1")\n')

print("Circle region file saved")

reg_point_path = outpath.replace(".cat","_points.reg")

with open(reg_point_path,"w") as f:

    f.write("# Region file format: DS9 version 4.1\n")
    f.write("global color=cyan point=cross 8 width=2\n")
    f.write("fk5\n")

    for _,row in lbg_candidates.iterrows():

        f.write(f'point({row["RA"]},{row["DEC"]})\n')

print("Point region file saved")

i shape: (239300, 12)
z shape: (297003, 12)
y shape: (286091, 12)
Y catalog size (before FLAG cleaning): 286091
After FLAG filtering:
i catalog: 239199
z catalog: 296872
y catalog: 286008
After SNR cuts: 3732 candidates
Removed near bright stars: 3
Final candidates: 3650
Saved to /Users/aishwarya/Desktop/CDFS_candidates_ZY_degraded.cat
Circle region file saved
Point region file saved


# match with pannstar

In [21]:
import numpy as np
import pandas as pd
from astropy.coordinates import SkyCoord
import astropy.units as u


cand_file = "/Users/aishwarya/Documents/Lyman_alpha/CDFS_candidates_ZY_degraded.cat"
matched_file = "/Users/aishwarya/Desktop/new_cdfs/cat_matched/LBG_I_CDFS_depth_decam_matched.cat"


# load candidate file (header already present)
cand_df = pd.read_csv(cand_file)


# load matched catalog
matched_df = pd.read_csv(matched_file)


# remove rows with invalid coordinates
cand_df = cand_df.dropna(subset=["RA","DEC"])
matched_df = matched_df.dropna(subset=["RA_CORRECTED","DEC_CORRECTED"])


# create coordinates
cand_coords = SkyCoord(
    cand_df["RA"].values * u.deg,
    cand_df["DEC"].values * u.deg
)

matched_coords = SkyCoord(
    matched_df["RA_CORRECTED"].values * u.deg,
    matched_df["DEC_CORRECTED"].values * u.deg
)


# match catalogs
idx, sep2d, _ = cand_coords.match_to_catalog_sky(matched_coords)


match_radius = 1  # arcsec (better for corrected astrometry)


matched_mask = sep2d.arcsec < match_radius


# remove matches
clean_df = cand_df[~matched_mask]


output_file = "/Users/aishwarya/Documents/Lyman_alpha/CDFS_candidates_removed_I.csv"

clean_df.to_csv(output_file, index=False)


print("Original candidates:", len(cand_df))
print("Removed matches:", np.sum(matched_mask))
print("Remaining candidates:", len(clean_df))
print("Saved to:", output_file)

Original candidates: 5753
Removed matches: 105
Remaining candidates: 5648
Saved to: /Users/aishwarya/Documents/Lyman_alpha/CDFS_candidates_removed_I.csv


In [31]:
import numpy as np
import pandas as pd
from astropy.coordinates import SkyCoord
import astropy.units as u

# ======================================================
# FILES
# ======================================================

cand_file = "/Users/aishwarya/Desktop/CDFS_candidates_ZY_degraded.cat"

matched_files = [
"/Users/aishwarya/Desktop/new_cdfs/cat_matched/LBG_Y_I_CDFS_decam_matched.cat",
"/Users/aishwarya/Desktop/new_cdfs/cat_matched/LBG_Y_Z_CDFS_decam_matched.cat",
"/Users/aishwarya/Desktop/new_cdfs/cat_matched/LBG_Z_CDFS_depth_decam_matched.cat"
]

output_file = "/Users/aishwarya/Documents/Lyman_alpha/CDFS_candidates_removed_matches.csv"

# ======================================================
# LOAD CANDIDATES
# ======================================================

cand_df = pd.read_csv(cand_file)

cand_df = cand_df.dropna(subset=["RA","DEC"])

# ======================================================
# LOAD AND COMBINE MATCHED CATALOGS
# ======================================================

matched_list = []

for file in matched_files:
    df = pd.read_csv(file)
    df = df.dropna(subset=["RA_CORRECTED","DEC_CORRECTED"])
    matched_list.append(df)

matched_df = pd.concat(matched_list, ignore_index=True)

# ======================================================
# CREATE SKY COORDINATES
# ======================================================

cand_coords = SkyCoord(
    cand_df["RA"].values * u.deg,
    cand_df["DEC"].values * u.deg
)

matched_coords = SkyCoord(
    matched_df["RA_CORRECTED"].values * u.deg,
    matched_df["DEC_CORRECTED"].values * u.deg
)

# ======================================================
# MATCH CATALOGS
# ======================================================

idx, sep2d, _ = cand_coords.match_to_catalog_sky(matched_coords)

match_radius = 2  # arcsec

matched_mask = sep2d.arcsec < match_radius

# ======================================================
# REMOVE MATCHES
# ======================================================

clean_df = cand_df[~matched_mask]

# ======================================================
# SAVE RESULT
# ======================================================

clean_df.to_csv(output_file, index=False)

print("Original candidates:", len(cand_df))
print("Removed matches:", np.sum(matched_mask))
print("Remaining candidates:", len(clean_df))
print("Saved to:", output_file)

Original candidates: 3653
Removed matches: 406
Remaining candidates: 3247
Saved to: /Users/aishwarya/Documents/Lyman_alpha/CDFS_candidates_removed_matches.csv


In [15]:
cand_file = "/Users/aishwarya/Documents/Lyman_alpha/CDFS_candidates_ZY_degraded.cat"

import pandas as pd

df = pd.read_csv(cand_file, header=None)
print(df.head())
print("Shape:", df.shape)

           0            1        2                   3        4   \
0          RA          DEC    Y_mag               Y_err    Z_mag   
1  52.8883086  -29.0125194  19.8408  0.0133794618725866  25.5522   
2  53.1126508  -29.0163836  22.0533  0.0615614327318654  23.7923   
3  53.0856956   -29.017243   23.629  0.2356719117756717  25.2952   
4  53.0187351  -28.9905838  20.3955  0.0178370401132026  21.8239   

                   5                   6      7      8                    9   \
0               Z_err               Z_SNR  I_mag  I_err                I_SNR   
1  0.1259635264669896    8.61944909936021  27.84    0.0  0.48593942838207765   
2  0.0815981004680868  13.305900487018853  27.84    0.0    1.370688500512466   
3  0.0661211010192661  16.420419321840573  27.84    0.0                  0.0   
4  0.0141760361173354   76.58954843028505  27.84    0.0                  0.0   

                   10                  11          12  
0                 Z-Y                 I-Z  I_detected 

#  code that matches to pannstar data and remove those stars if det

In [59]:
import numpy as np
import pandas as pd
from astropy.coordinates import SkyCoord
import astropy.units as u

# ============================================================
# FILE PATHS
# ============================================================

file1 = "/Users/aishwarya/Documents/Lyman_alpha/CDFS_candidates_ZY_degraded.cat"
file2 = "/Users/aishwarya/Desktop/new_cdfs/cat_matched/LBG_Y_CDFS_depth_decam_matched.cat"

output_file = "/Users/aishwarya/Documents/Lyman_alpha/CDFS_candidates_ZY_cleaned.cat"

# ============================================================
# READ CORRECTLY (COMMA SEPARATED!)
# ============================================================

cat1 = pd.read_csv(file1, sep=",")
cat2 = pd.read_csv(file2, sep=",")

# ============================================================
# DISPLAY FIRST FEW LINES + COLUMNS
# ============================================================

print("\n========== CATALOG 1 ==========")
print(cat1.head())
print("\nColumns in Catalog 1:")
print(cat1.columns)

print("\n========== CATALOG 2 ==========")
print(cat2.head())
print("\nColumns in Catalog 2:")
print(cat2.columns)

# ============================================================
# RA/DEC COLUMN NAMES (FROM YOUR OUTPUT)
# ============================================================

ra1 = "RA"
dec1 = "DEC"

ra2 = "ALPHA_J2000"
dec2 = "DELTA_J2000"

# ============================================================
# CREATE SKYCOORD OBJECTS
# ============================================================

coords1 = SkyCoord(cat1[ra1].values * u.deg,
                   cat1[dec1].values * u.deg)

coords2 = SkyCoord(cat2[ra2].values * u.deg,
                   cat2[dec2].values * u.deg)

# ============================================================
# MATCH
# ============================================================

idx, d2d, _ = coords1.match_to_catalog_sky(coords2)

match_radius = 1.0 * u.arcsec
match_mask = d2d < match_radius

print(f"\nNumber of matches found: {np.sum(match_mask)}")

# ============================================================
# REMOVE MATCHED OBJECTS
# ============================================================

clean_cat1 = cat1[~match_mask]

print(f"Original size: {len(cat1)}")
print(f"Final size after removal: {len(clean_cat1)}")

# ============================================================
# SAVE CLEANED CATALOG
# ============================================================

clean_cat1.to_csv(output_file, sep=",", index=False)

print("\nCleaned catalog saved.")


========== CATALOG 1 ==========
          RA        DEC    Y_mag     Y_err    Z_mag     Z_err      Z_SNR  \
0  53.017475 -29.004511  23.5375  0.009425  24.8687  0.028185  38.521392   
1  53.103500 -29.010934  21.4108  0.009045  23.7613  0.011330  95.831760   
2  52.965937 -29.004577  22.0908  0.009080  24.4763  0.020788  52.227837   
3  52.942330 -29.007272  22.0841  0.009108  24.2765  0.016784  64.690217   
4  52.960819 -29.001036  21.8529  0.009055  23.8024  0.011799  92.021457   

   I_mag  I_err     I_SNR     Z-Y     I-Z  I_detected  
0  27.84    0.0  0.010967  1.3312  2.9713        True  
1  27.84    0.0  0.010967  2.3505  4.0787        True  
2  27.84    0.0  0.010967  2.3855  3.3637        True  
3  27.84    0.0  1.435000  2.1924  3.5635        True  
4  27.84    0.0  0.010967  1.9495  4.0376        True  

Columns in Catalog 1:
Index(['RA', 'DEC', 'Y_mag', 'Y_err', 'Z_mag', 'Z_err', 'Z_SNR', 'I_mag',
       'I_err', 'I_SNR', 'Z-Y', 'I-Z', 'I_detected'],
      dtype='object')



In [61]:
import numpy as np
import pandas as pd
from astropy.coordinates import SkyCoord
import astropy.units as u

# ============================================================
# FILE PATHS
# ============================================================

file1 = "/Users/aishwarya/Documents/Lyman_alpha/CDFS_candidates_ZY_degraded.cat"
file2 = "/Users/aishwarya/Desktop/new_cdfs/cat_matched/LBG_Y_CDFS_depth_decam_matched.cat"

output_file = "/Users/aishwarya/Documents/Lyman_alpha/CDFS_candidates_ZY_cleaned.cat"

# ============================================================
# READ FILES
# ============================================================

cat1 = pd.read_csv(file1, sep=",")
cat2 = pd.read_csv(file2, sep=",")

print("\nCatalog 1 size:", len(cat1))
print("Catalog 2 size:", len(cat2))

# ============================================================
# STAR SELECTION IN CATALOG 2
# Using |MAG_APER - MAG_AUTO| < 0.1
# ============================================================

morph_diff = np.abs(cat2["MAG_APER"] - cat2["MAG_AUTO"])

star_mask = morph_diff < 0.1

stars_cat2 = cat2[star_mask].copy()

print("\nNumber of stars identified in catalog 2:", len(stars_cat2))

# ============================================================
# SKYCOORD OBJECTS
# ============================================================

coords1 = SkyCoord(cat1["RA"].values * u.deg,
                   cat1["DEC"].values * u.deg)

coords2 = SkyCoord(stars_cat2["ALPHA_J2000"].values * u.deg,
                   stars_cat2["DELTA_J2000"].values * u.deg)

# ============================================================
# MATCH ONLY TO STARS
# ============================================================

idx, d2d, _ = coords1.match_to_catalog_sky(coords2)

match_radius = 1.0 * u.arcsec
match_mask = d2d < match_radius

print("\nNumber of stellar matches found:", np.sum(match_mask))

# ============================================================
# REMOVE ONLY STELLAR MATCHES
# ============================================================

clean_cat1 = cat1[~match_mask].copy()

print("\nOriginal size:", len(cat1))
print("Final size after removing stars:", len(clean_cat1))

# ============================================================
# SAVE CLEANED CATALOG
# ============================================================

clean_cat1.to_csv(output_file, sep=",", index=False)

print("\nCleaned catalog saved to:")
print(output_file)


Catalog 1 size: 414
Catalog 2 size: 81418

Number of stars identified in catalog 2: 7373

Number of stellar matches found: 3

Original size: 414
Final size after removing stars: 411

Cleaned catalog saved to:
/Users/aishwarya/Documents/Lyman_alpha/CDFS_candidates_ZY_cleaned.cat


In [33]:
import numpy as np
import pandas as pd
from astropy.coordinates import SkyCoord
import astropy.units as u

# ============================================================
# File paths
# ============================================================
i_file = "/Users/aishwarya/Desktop/Lyman_alpha_2/CAT/i_band_degraded.cat"
z_file = "/Users/aishwarya/Desktop/Lyman_alpha_2/CAT/z_band_degraded.cat"
y_file = "/Users/aishwarya/Desktop/new_cdfs/cat/LBG_Y_Y_CDFS.cat"
redlist_file = "/Users/aishwarya/Desktop/comparision/Not_Source.txt"

# ============================================================
# READ ALL FILES
# ============================================================
i_df = pd.read_csv(i_file, sep=r"\s+", comment="#", header=None)
z_df = pd.read_csv(z_file, sep=r"\s+", comment="#", header=None)
y_df = pd.read_csv(y_file, sep=r"\s+", comment="#", header=None)

# ============================================================
# ASSIGN COLUMN NAMES
# ============================================================
i_df.columns = [
    "NUMBER", "X_IMAGE", "Y_IMAGE", "RA", "DEC",
    "MAG_ORIG", "MAGERR_ORIG",
    "MAG_APER", "MAGERR_APER",
    "FLAGS",
    "MAG_DEGRADED", "MAGERR_DEGRADED"
]

z_df.columns = [
    "NUMBER", "X_IMAGE", "Y_IMAGE", "RA", "DEC",
    "MAG_ORIG", "MAGERR_ORIG",
    "MAG_APER", "MAGERR_APER",
    "FLAGS",
    "MAG_DEGRADED", "MAGERR_DEGRADED"
]

y_df.columns = [
    "NUMBER",
    "X_IMAGE",
    "Y_IMAGE",
    "ALPHA_J2000",
    "DELTA_J2000",
    "MAG_APER",
    "MAGERR_APER",
    "MAG_AUTO",
    "MAGERR_AUTO",
    "FLAGS"
]

# ============================================================
# Use degraded magnitudes (i & z)
# ============================================================
i_df["MAG_CAL"] = i_df["MAG_DEGRADED"]
i_df["MAGERR_CAL"] = i_df["MAGERR_DEGRADED"]

z_df["MAG_CAL"] = z_df["MAG_DEGRADED"]
z_df["MAGERR_CAL"] = z_df["MAGERR_DEGRADED"]

# ============================================================
# Y-band calibration
# ============================================================
ZP_Y = 30.3382
ZPERR_Y = 0.009

y_df["MAG_CAL"] = y_df["MAG_APER"] + ZP_Y
y_df["MAGERR_CAL"] = np.sqrt(y_df["MAGERR_APER"]**2 + ZPERR_Y**2)

# ============================================================
# SNR function
# ============================================================
def magerr_to_snr(err):
    snr = 2.5 / (np.log(10) * err)
    snr[~np.isfinite(snr)] = 0.0
    return snr

# ============================================================
# Coordinate matching
# ============================================================
y_coords = SkyCoord(
    y_df["ALPHA_J2000"].values,
    y_df["DELTA_J2000"].values,
    unit="deg"
)

i_coords = SkyCoord(i_df["RA"].values, i_df["DEC"].values, unit="deg")
z_coords = SkyCoord(z_df["RA"].values, z_df["DEC"].values, unit="deg")

match_radius = 1.0 * u.arcsec

idx_i, sep_i, _ = y_coords.match_to_catalog_sky(i_coords)
idx_z, sep_z, _ = y_coords.match_to_catalog_sky(z_coords)

def matched_array(src_df, idx, sep):
    n = len(y_df)
    mag = np.full(n, np.nan)
    err = np.full(n, np.nan)
    detected = sep < match_radius

    mag[detected] = src_df["MAG_CAL"].values[idx[detected]]
    err[detected] = src_df["MAGERR_CAL"].values[idx[detected]]

    return mag, err, detected

# ============================================================
# Matched photometry
# ============================================================
i_mag, i_err, i_detected = matched_array(i_df, idx_i, sep_i)
z_mag, z_err, z_detected = matched_array(z_df, idx_z, sep_z)

y_mag = y_df["MAG_CAL"].values
y_err = y_df["MAGERR_CAL"].values

# ============================================================
# Compute SNR
# ============================================================
i_snr = magerr_to_snr(i_err)
z_snr = magerr_to_snr(z_err)
y_snr = magerr_to_snr(y_err)

# ============================================================
# Colors
# ============================================================
color_zy = z_mag - y_mag
color_iz = i_mag - z_mag
color_zy_err = np.sqrt(z_err**2 + y_err**2)

# ============================================================
# Selection cuts (NO forced I replacement)
# ============================================================
cut_z_detected = z_detected
cut_z_snr      = z_snr > 4.0
cut_y_snr      = y_snr > 4.0

cut_zy         = color_zy > 0.8
cut_break      = color_iz > 1.0

# True non-detection in I
cut_i_nondet   = i_snr < 2.0

cut_sig        = np.abs(color_zy) > 2.5 * color_zy_err

# ============================================================
# Final selection
# ============================================================
final_sel = (
    cut_z_detected &
    cut_z_snr &
    cut_y_snr &
    cut_zy &
    cut_break &
    cut_i_nondet &
    cut_sig
)

lbg_idx = np.where(final_sel)[0]
print(f"After selection: {len(lbg_idx)} candidates")

# ============================================================
# Build LBG catalog
# ============================================================
lbg_candidates = pd.DataFrame({
    "RA": y_df["ALPHA_J2000"].values[lbg_idx],
    "DEC": y_df["DELTA_J2000"].values[lbg_idx],
    "Y_mag": y_mag[lbg_idx],
    "Z_mag": z_mag[lbg_idx],
    "I_mag": i_mag[lbg_idx],
    "Z-Y": color_zy[lbg_idx],
    "I-Z": color_iz[lbg_idx],
    "Z_SNR": z_snr[lbg_idx],
    "I_SNR": i_snr[lbg_idx]
})

# ============================================================
# Remove red-list sources
# ============================================================
red_df = pd.read_csv(redlist_file, sep=r"\s+", names=["RA", "DEC"])
red_coords = SkyCoord(red_df["RA"].values, red_df["DEC"].values, unit="deg")

lbg_coords = SkyCoord(
    lbg_candidates["RA"].values,
    lbg_candidates["DEC"].values,
    unit="deg"
)

idx_red, sep_red, _ = lbg_coords.match_to_catalog_sky(red_coords)
lbg_candidates = lbg_candidates[sep_red.arcsec > 1.0]

# ============================================================
# Output
# ============================================================
outpath = "/Users/aishwarya/Documents/Lyman_alpha/CDFS_candidates_ZY_degraded.cat"
lbg_candidates.to_csv(outpath, index=False)

print(f"Final LBG candidates: {len(lbg_candidates)}")
print(f"Saved to {outpath}")

After selection: 69 candidates
Final LBG candidates: 69
Saved to /Users/aishwarya/Documents/Lyman_alpha/CDFS_candidates_ZY_degraded.cat


In [19]:
import numpy as np
import pandas as pd
from astropy.coordinates import SkyCoord
import astropy.units as u

# ============================================================
# Input files
# ============================================================
i_file = "/Users/aishwarya/Desktop/new_cdfs/cat/LBG_Y_I_CDFS.cat"
z_file = "/Users/aishwarya/Desktop/new_cdfs/cat/LBG_Y_Z_CDFS.cat"
y_file = "/Users/aishwarya/Desktop/new_cdfs/cat/LBG_Y_Y_CDFS.cat"

redlist_file = "/Users/aishwarya/Desktop/comparision/Not_Source.txt"

colnames = [
    "ID", "X", "Y", "RA", "DEC",
    "MAG_APER", "MAGERR_APER",
    "MAG_AUTO", "MAGERR_AUTO", "FLAGS"
]

i_df = pd.read_csv(i_file, sep=r"\s+", comment="#", names=colnames)
z_df = pd.read_csv(z_file, sep=r"\s+", comment="#", names=colnames)
y_df = pd.read_csv(y_file, sep=r"\s+", comment="#", names=colnames)

print(f"Y catalog size: {len(y_df)}")

# ============================================================
# Zeropoints
# ============================================================
ZP = {
    "i": (31.354,  0.004),
    "z": (31.524,  0.004),
    "y": (30.3382, 0.009)
}

def calibrate(df, band):
    zp, zp_err = ZP[band]
    df["MAG_CAL"] = df["MAG_APER"] + zp
    df["MAGERR_CAL"] = np.sqrt(df["MAGERR_APER"]**2 + zp_err**2)

calibrate(i_df, "i")
calibrate(z_df, "z")
calibrate(y_df, "y")

# ============================================================
# SNR conversion
# ============================================================
def magerr_to_snr(err):
    snr = 2.5 / (np.log(10) * err)
    snr[~np.isfinite(snr)] = 0.0
    return snr

# ============================================================
# Coordinate matching (Y reference)
# ============================================================
y_coords = SkyCoord(y_df.RA.values, y_df.DEC.values, unit="deg")
i_coords = SkyCoord(i_df.RA.values, i_df.DEC.values, unit="deg")
z_coords = SkyCoord(z_df.RA.values, z_df.DEC.values, unit="deg")

match_radius = 1.0 * u.arcsec

idx_i, sep_i, _ = y_coords.match_to_catalog_sky(i_coords)
idx_z, sep_z, _ = y_coords.match_to_catalog_sky(z_coords)

def matched_array(src_df, idx, sep):
    n = len(y_df)
    mag = np.full(n, np.nan)
    err = np.full(n, np.nan)
    detected = sep < match_radius

    mag[detected] = src_df.MAG_CAL.values[idx[detected]]
    err[detected] = src_df.MAGERR_CAL.values[idx[detected]]

    return mag, err, detected

# ============================================================
# Matched photometry
# ============================================================
i_mag, i_err, i_detected = matched_array(i_df, idx_i, sep_i)
z_mag, z_err, z_detected = matched_array(z_df, idx_z, sep_z)

y_mag = y_df.MAG_CAL.values
y_err = y_df.MAGERR_CAL.values

# ============================================================
# SNR
# ============================================================
i_snr = magerr_to_snr(i_err)
z_snr = magerr_to_snr(z_err)
y_snr = magerr_to_snr(y_err)

# ============================================================
# Proper I-band upper limit handling
# ============================================================
I_LIMIT = 27.84

i_nondet = (~i_detected) | (i_snr < 2.0)

# Copy magnitudes for color calculation only
i_mag_for_color = np.copy(i_mag)
i_mag_for_color[i_nondet] = I_LIMIT

# ============================================================
# Colors
# ============================================================
color_zy = z_mag - y_mag
color_iz = i_mag_for_color - z_mag
color_zy_err = np.sqrt(z_err**2 + y_err**2)

# ============================================================
# Selection cuts
# ============================================================
cut_z_detected = z_detected
cut_z_snr      = z_snr > 4.0
cut_y_snr      = y_snr > 4.0

cut_zy         = color_zy > 0.8
cut_sig        = np.abs(color_zy) > 2.5 * color_zy_err

# Proper break treatment:
cut_break = (
    (i_detected & (i_snr >= 2.0) & (color_iz > 1.0)) |
    (i_nondet & ((I_LIMIT - z_mag) > 1.0))
)

cut_i_dropout  = i_nondet

# ============================================================
# Final selection
# ============================================================
final_sel = (
    cut_z_detected &
    cut_z_snr &
    cut_y_snr &
    cut_zy &
    cut_break &
    cut_i_dropout &
    cut_sig
)

lbg_idx = np.where(final_sel)[0]
print(f"After selection: {len(lbg_idx)} candidates")

# ============================================================
# Build LBG catalog
# ============================================================
lbg_candidates = pd.DataFrame({
    "RA": y_df.RA.values[lbg_idx],
    "DEC": y_df.DEC.values[lbg_idx],
    "Y_mag": y_mag[lbg_idx],
    "Y_err": y_err[lbg_idx],
    "Z_mag": z_mag[lbg_idx],
    "Z_err": z_err[lbg_idx],
    "Z_SNR": z_snr[lbg_idx],
    "I_mag": i_mag_for_color[lbg_idx],
    "I_SNR": i_snr[lbg_idx],
    "Z-Y": color_zy[lbg_idx],
    "I-Z": color_iz[lbg_idx],
    "I_detected": i_detected[lbg_idx]
})

# ============================================================
# Remove red-list sources
# ============================================================
red_df = pd.read_csv(redlist_file, sep=r"\s+", names=["RA", "DEC"])
red_coords = SkyCoord(red_df.RA.values, red_df.DEC.values, unit="deg")

lbg_coords = SkyCoord(lbg_candidates.RA.values,
                      lbg_candidates.DEC.values,
                      unit="deg")

idx_red, sep_red, _ = lbg_coords.match_to_catalog_sky(red_coords)

lbg_candidates = lbg_candidates[sep_red.arcsec > 1.0]

# ============================================================
# Output
# ============================================================
outpath = "/Users/aishwarya/Documents/Lyman_alpha/CDFS_candidates_ZY.cat"
lbg_candidates.to_csv(outpath, index=False)

print(f"Final LBG candidates after red-list removal: {len(lbg_candidates)}")
print(f"Saved to {outpath}")

print("\nFinal candidates:")
for _, row in lbg_candidates.iterrows():
    print(
        f"RA={row.RA:.6f}, DEC={row.DEC:.6f}, "
        f"I_SNR={row.I_SNR:.2f}, "
        f"I_mag={row.I_mag:.2f}, "
        f"z_mag={row.Z_mag:.2f}, y_mag={row.Y_mag:.2f}"
    )

Y catalog size: 295284
After selection: 6289 candidates
Final LBG candidates after red-list removal: 6289
Saved to /Users/aishwarya/Documents/Lyman_alpha/CDFS_candidates_ZY.cat

Final candidates:
RA=52.253510, DEC=-28.974876, I_SNR=0.00, I_mag=27.84, z_mag=21.37, y_mag=19.77
RA=52.273561, DEC=-28.989838, I_SNR=0.00, I_mag=27.84, z_mag=25.28, y_mag=17.85
RA=52.200363, DEC=-28.972302, I_SNR=0.27, I_mag=27.84, z_mag=21.73, y_mag=19.15
RA=52.253307, DEC=-28.978792, I_SNR=0.01, I_mag=27.84, z_mag=21.75, y_mag=20.32
RA=52.266791, DEC=-28.989961, I_SNR=0.00, I_mag=27.84, z_mag=25.22, y_mag=17.83
RA=52.270249, DEC=-28.984956, I_SNR=0.00, I_mag=27.84, z_mag=20.83, y_mag=17.83
RA=52.270218, DEC=-29.003481, I_SNR=0.00, I_mag=27.84, z_mag=21.51, y_mag=18.81
RA=53.017520, DEC=-29.013835, I_SNR=1.73, I_mag=27.84, z_mag=22.07, y_mag=19.08
RA=53.021385, DEC=-29.013834, I_SNR=1.45, I_mag=27.84, z_mag=24.11, y_mag=18.99
RA=53.017714, DEC=-29.004641, I_SNR=0.00, I_mag=27.84, z_mag=25.89, y_mag=19.82
RA=5

In [55]:
import pandas as pd
import os


def cat_to_reg(cat_file,
               ra_col="RA",
               dec_col="DEC",
               point_style="circle",
               point_size=6,
               color="cyan",
               radius_arcsec=1.0):
    """
    Convert a catalog to DS9 region files:
    - point markers
    - 1 arcsec radius circles
    (NO labels)
    """

    base = cat_file.replace(".cat", "")
    reg_point = base + "_points.reg"
    reg_circle = base + "_r1arcsec.reg"

    df = pd.read_csv(cat_file)

    # ---------- POINT REGIONS ----------
    with open(reg_point, "w", encoding="utf8") as f:
        f.write("# Region file format: DS9 version 4.1\n")
        f.write(f"global color={color} width=2\n")
        f.write("fk5\n")

        for _, row in df.iterrows():
            f.write(
                f"point({row[ra_col]},{row[dec_col]}) "
                f"# point={point_style} {point_size}\n"
            )

    # ---------- 1 ARCSEC CIRCLES ----------
    with open(reg_circle, "w", encoding="utf8") as f:
        f.write("# Region file format: DS9 version 4.1\n")
        f.write(f"global color={color} width=2\n")
        f.write("fk5\n")

        for _, row in df.iterrows():
            f.write(
                f'circle({row[ra_col]},{row[dec_col]},{radius_arcsec}")\n'
            )

    print(f"Created:")
    print(f"  {reg_point}")
    print(f"  {reg_circle}")


if __name__ == "__main__":

    CAT_FILES = [
        "/Users/aishwarya/Documents/Lyman_alpha/CDFS_candidates_ZY_degraded.cat"
    ]

    for cat in CAT_FILES:
        if not os.path.exists(cat):
            print(f"File not found: {cat}")
            continue

        cat_to_reg(
            cat_file=cat,
            ra_col="RA",
            dec_col="DEC",
            point_style="circle",
            point_size=6,
            color="cyan",
            radius_arcsec=1.0
        )

    print("All catalogs converted to DS9 region files.")


Created:
  /Users/aishwarya/Documents/Lyman_alpha/CDFS_candidates_ZY_degraded_points.reg
  /Users/aishwarya/Documents/Lyman_alpha/CDFS_candidates_ZY_degraded_r1arcsec.reg
All catalogs converted to DS9 region files.


# CHECK

In [47]:
import numpy as np
import pandas as pd
from astropy.coordinates import SkyCoord
import astropy.units as u

# ============================================================
# File paths
# ============================================================
i_file = "/Users/aishwarya/Desktop/Lyman_alpha_2/CAT/i_band_degraded.cat"
z_file = "/Users/aishwarya/Desktop/Lyman_alpha_2/CAT/z_band_degraded.cat"
y_file = "/Users/aishwarya/Desktop/new_cdfs/cat/LBG_Y_Y_CDFS.cat"
redlist_file = "/Users/aishwarya/Downloads/candidates_red_list_cdfs.txt"

# ============================================================
# READ ALL FILES (NO HEADER)
# ============================================================
i_df = pd.read_csv(i_file, sep=r"\s+", comment="#", header=None)
z_df = pd.read_csv(z_file, sep=r"\s+", comment="#", header=None)
y_df = pd.read_csv(y_file, sep=r"\s+", comment="#", header=None)

print("i shape:", i_df.shape)
print("z shape:", z_df.shape)
print("y shape:", y_df.shape)

# ============================================================
# ASSIGN COLUMN NAMES
# ============================================================
i_df.columns = [
    "NUMBER", "X_IMAGE", "Y_IMAGE", "RA", "DEC",
    "MAG_ORIG", "MAGERR_ORIG",
    "MAG_APER", "MAGERR_APER",
    "FLAGS",
    "MAG_DEGRADED", "MAGERR_DEGRADED"
]

z_df.columns = [
    "NUMBER", "X_IMAGE", "Y_IMAGE", "RA", "DEC",
    "MAG_ORIG", "MAGERR_ORIG",
    "MAG_APER", "MAGERR_APER",
    "FLAGS",
    "MAG_DEGRADED", "MAGERR_DEGRADED"
]

y_df.columns = [
    "NUMBER",
    "X_IMAGE",
    "Y_IMAGE",
    "ALPHA_J2000",
    "DELTA_J2000",
    "MAG_APER",
    "MAGERR_APER",
    "MAG_AUTO",
    "MAGERR_AUTO",
    "FLAGS"
]

print(f"Y catalog size: {len(y_df)}")

# ============================================================
# USE DEGRADED MAGNITUDES (i & z)
# ============================================================
i_df["MAG_CAL"] = i_df["MAG_DEGRADED"]
i_df["MAGERR_CAL"] = i_df["MAGERR_DEGRADED"]

z_df["MAG_CAL"] = z_df["MAG_DEGRADED"]
z_df["MAGERR_CAL"] = z_df["MAGERR_DEGRADED"]

# ============================================================
# Y-band calibration
# ============================================================
ZP_Y = 30.3382
ZPERR_Y = 0.009

y_df["MAG_CAL"] = y_df["MAG_APER"] + ZP_Y
y_df["MAGERR_CAL"] = np.sqrt(y_df["MAGERR_APER"]**2 + ZPERR_Y**2)

# ============================================================
# SNR from magnitude error
# ============================================================
def magerr_to_snr(err):
    snr = 2.5 / (np.log(10) * err)
    snr[~np.isfinite(snr)] = 0.0
    return snr

# ============================================================
# Coordinate matching (Y is reference)
# ============================================================
y_coords = SkyCoord(
    y_df["ALPHA_J2000"].values,
    y_df["DELTA_J2000"].values,
    unit="deg"
)

i_coords = SkyCoord(i_df["RA"].values, i_df["DEC"].values, unit="deg")
z_coords = SkyCoord(z_df["RA"].values, z_df["DEC"].values, unit="deg")

match_radius = 1.0 * u.arcsec

idx_i, sep_i, _ = y_coords.match_to_catalog_sky(i_coords)
idx_z, sep_z, _ = y_coords.match_to_catalog_sky(z_coords)

# ============================================================
# Matched photometry WITH FLAGS
# ============================================================
def matched_array(src_df, idx, sep):
    n = len(y_df)
    mag = np.full(n, np.nan)
    err = np.full(n, np.nan)
    flags = np.full(n, 99)   # default bad flag

    detected = sep < match_radius

    mag[detected] = src_df["MAG_CAL"].values[idx[detected]]
    err[detected] = src_df["MAGERR_CAL"].values[idx[detected]]
    flags[detected] = src_df["FLAGS"].values[idx[detected]]

    return mag, err, flags, detected


i_mag, i_err, i_flags, i_detected = matched_array(i_df, idx_i, sep_i)
z_mag, z_err, z_flags, z_detected = matched_array(z_df, idx_z, sep_z)

y_mag = y_df["MAG_CAL"].values
y_err = y_df["MAGERR_CAL"].values
y_flags = y_df["FLAGS"].values

# ============================================================
# Compute SNR
# ============================================================
i_snr = magerr_to_snr(i_err)
z_snr = magerr_to_snr(z_err)
y_snr = magerr_to_snr(y_err)

# ============================================================
# FORCE I-band non-detections
# ============================================================
I_LIMIT = 27.84
i_nondet = (~i_detected) | (i_snr < 2.0)

i_mag[i_nondet] = I_LIMIT
i_err[i_nondet] = 0.0

# ============================================================
# Colors
# ============================================================
color_zy = z_mag - y_mag
color_iz = i_mag - z_mag
color_zy_err = np.sqrt(z_err**2 + y_err**2)

# ============================================================
# FLAGS CUTS  (CRITICAL FIX)
# ============================================================
cut_z_flags = z_flags == 0
cut_i_flags = i_flags == 0
cut_y_flags = y_flags == 0

# ============================================================
# Selection cuts
# ============================================================
cut_z_detected = z_detected
cut_z_snr      = z_snr > 4.0
cut_y_snr      = y_snr > 4.0

cut_zy         = color_zy > 0.8
cut_break      = color_iz > 1.0

cut_i_dropout  = i_nondet
cut_sig        = np.abs(color_zy) > 2.5 * color_zy_err

# ============================================================
# Final selection
# ============================================================
final_sel = (
    cut_z_detected &
    cut_z_snr &
    cut_y_snr &
    cut_zy &
    cut_break &
    cut_i_dropout &
    cut_sig &
    cut_z_flags &
    cut_i_flags &
    cut_y_flags
)

lbg_idx = np.where(final_sel)[0]
print(f"After full cuts (including FLAGS): {len(lbg_idx)} candidates")

# ============================================================
# Build LBG catalog
# ============================================================
lbg_candidates = pd.DataFrame({
    "RA": y_df["ALPHA_J2000"].values[lbg_idx],
    "DEC": y_df["DELTA_J2000"].values[lbg_idx],
    "Y_mag": y_mag[lbg_idx],
    "Y_err": y_err[lbg_idx],
    "Z_mag": z_mag[lbg_idx],
    "Z_err": z_err[lbg_idx],
    "Z_SNR": z_snr[lbg_idx],
    "I_mag": i_mag[lbg_idx],
    "I_err": i_err[lbg_idx],
    "I_SNR": i_snr[lbg_idx],
    "Z-Y": color_zy[lbg_idx],
    "I-Z": color_iz[lbg_idx],
    "I_detected": i_detected[lbg_idx]
})

# ============================================================
# Remove red-list sources
# ============================================================
red_df = pd.read_csv(redlist_file, sep=r"\s+", names=["RA", "DEC"])
red_coords = SkyCoord(red_df["RA"].values, red_df["DEC"].values, unit="deg")

lbg_coords = SkyCoord(
    lbg_candidates["RA"].values,
    lbg_candidates["DEC"].values,
    unit="deg"
)

idx_red, sep_red, _ = lbg_coords.match_to_catalog_sky(red_coords)
lbg_candidates = lbg_candidates[sep_red.arcsec > 1.0]

# ============================================================
# Output
# ============================================================
outpath = "/Users/aishwarya/Documents/Lyman_alpha/CDFS_candidates_ZY_degraded.cat"
lbg_candidates.to_csv(outpath, index=False)

print(f"Final LBG candidates after red-list removal: {len(lbg_candidates)}")
print(f"Saved to {outpath}")

i shape: (245186, 12)
z shape: (245186, 12)
y shape: (245186, 10)
Y catalog size: 245186
After full cuts (including FLAGS): 997 candidates
Final LBG candidates after red-list removal: 994
Saved to /Users/aishwarya/Documents/Lyman_alpha/CDFS_candidates_ZY_degraded.cat
